### Readme - _DataProcessing.ipynb_

#### This notebook creates a dataset for media measurement using Snowflake Tables and _.py_ modules. Some people may call it the ET part of ETL.

- Modules live in `./modules/`
- Each module contains a discrete step of the data creation process. 
- Steps are wrapped within callable functions to keep the notebook environment clean. 
- Each block in this notebook calls one of these functions which creates transient datasets in the folder where this code is run.
- Once complete, data can be loaded for modeling and analysis.

---

### Modules:
   
1. `read_snowflake_data.py` Accessess the necessary Snowflake tables and writes them to `./data/cache/` as _.parquet_ files.
2. `data_cleaner.py` Contains a master function _clean_all_data_ which calls sub-functions. These sub-functions each clean and transform a different cached _.parquet_ file and saves to `./data/clean/`
3. `merge_data.py` Combines the cleaned datasets into a single master panel which is saved to `./data/master.parquet`
4. `macro_tools.py` Pulls two macroeconomic controls - the **WEI** and **PCE** from the _FRED_ API. It also manages the cleaning, transformation and attachment of these controls to the master panel
5. `reverse_causality_controls.py` Somewhat of a work in progress, generally meant to handle reverse causality. 

---

### Known Issues:

- New media spending dataset has broken the chain, `merge_data` currently being fixed
- `macro_tools` was a vibe coded slopfest and is currently getting cleaned up.
    - `macro_tools.ipynb` is the live testing environment
    - The main script will be updated once testing is finished

- Revese Causality needs help, it's basically non-existent currently

- The GA dataset may be inaccurate, but no one around here really seems to know anymore

- The filestructure needs a lot of help

---

In [1]:
# --- Autoreload for Custom Modules ---

%load_ext autoreload
%autoreload 2

In [2]:
# --- Community Imports ---

import snowflake.snowpark as snowpark
from snowflake.snowpark.session import Session
import pandas as pd
import datetime as dt
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# --- Snowflake Credentials & Connector ---

connection_parameters = {
    "account": os.getenv("SNOWFLAKE_ACCOUNT"),
    "user": os.getenv("SNOWFLAKE_USER"),
    "authenticator": os.getenv("SNOWFLAKE_AUTHENTICATOR"),
    "insecure_mode":True
}

session = Session.builder.configs(connection_parameters).create()

Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://dish.okta.com/app/snowflake/exk9ewak82R5btDoW2p7/sso/saml?SAMLRequest=pZLLbtswEEV%2FRWDXEinFrxCWAzWGUwVJa9iKW3dHS7RNiCJVDmU5%2FfpSfhTpItl0R5B3eO7MnfHdsZLegRsQWsUoDAjyuMp1IdQuRi%2FZzB8hDyxTBZNa8Ri9ckB3kzGwStY0aexeLfivhoP13EcKaPcQo8YoqhkIoIpVHKjN6TJ5fqJRQCgD4MY6HLqUFCAca29tTTFu2zZobwJtdjgihGByi52qk3xCbxD1x4zaaKtzLa8lR9fTO4gQk16HcApHmF8KPwt1HsFHlM1ZBPRLls39%2Bbdlhrzk2t29VtBU3Cy5OYicvyyezgbAOVivkoiEN6OgAb91s%2FOjoDbiwCyXQpUBKN1uJSt5rqu6sQ4RuBPe8gJLvRNucOk0RnUpCpaSbJbD4CAehiFZlcdsu%2BHJ%2BnEvfxTr3%2BG8etykBS8eBpDmyFtdY466mFOAhqeqC9e6KxINfBL6pJ9FhEZ92hsGYb%2F3E3lTZ1AoZk%2BV1w4KAftAl5adnLG6xn9NY34sb3nLylG06G%2FsVH%2BP6iEG0LgLDp13h57oZvJfExnjt19ddvKriymdzrUU%2Bas306Zi9v0UwyA83YjC356klFdMyKQoDAdwaUqp23vDnY8YWdNwhCdn6r%2FLP%2FkD&RelayState=ver

## Snowflake Data Read-Ins
#### This is pretty self-explanatory

**Things to know:**

- Pulls in each dataset using the snowpark API
- Requires Snowflake credentials
- Grabs data if it's older than the specified hour age in the script
- _Writes to the uncleaned data cache_ `./data/cache/`


In [4]:
# --- Snowflake Data Read

from modules.read_snowflake_data import fetch_and_cache_data
fetch_and_cache_data(session=session)

All data caches are fresh. No queries needed.


## Data Cleaning

#### This script cleans and transforms each dataset

- Because of how markets work, we have to create a manual mapping. `market_map_os_to_bst` is effectively a giant dictionary that allows us to run the _.map_ function and join Opensignal market performance data to Boosts business-level outcomes

- Frankly, I've kind of lost track of what's going on here. I'm about 80% sure `market_map_ga_to_bst` is not getting used anymore, it was for robustness checking.
    - We're using population as a proxy for media investment by market to construct a _shift-share_. I did some correlation analysis, it works trust me.

- Writes cleaned to `./data/clean/`

In [5]:
# --- Initial Data Cleaning

from modules.data_cleaner import clean_all_data
clean_all_data()

Local Data Description


,Date,Digital,Festivals and Sponsorships,OOH,Other_Local,Radio
count,8603,8603.000000,8603.000000,8603.000000,8603.000000,8603.000000
mean,2025-02-13 13:54:24.396140800,7930.167692,3756.662252,95122.667572,3453.508895,21001.269788
min,2024-01-09 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2024-09-28 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2025-03-01 00:00:00,0.000000,0.000000,52670.000000,0.000000,10013.620000
75%,2025-06-17 00:00:00,5417.280000,1007.190000,150000.000000,0.000000,30009.780000
max,2026-01-27 00:00:00,137488.510000,130011.440000,668549.470000,114521.280000,263954.560000
std,NaN,17208.845974,10253.329988,113752.046648,14465.420428,30706.865543


National Data Description


/Users/Ben.Pharris/Documents/Code Gallery/Media Measurement/modules/data_cleaner.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df['Date'] = pd.to_datetime(cleaned_df['Date'])


,Date,Spend
count,133407,133407.000000
mean,2025-04-01 00:00:00.000000512,3247.133462
min,2024-07-01 00:00:00,0.000000
25%,2024-11-15 00:00:00,0.000000
50%,2025-04-01 00:00:00,0.000000
75%,2025-08-16 00:00:00,18.262500
max,2025-12-31 00:00:00,762752.874000
std,NaN,23055.057508


...

,Date,GA
count,203348,203348.000000
mean,2025-04-21 16:18:45.385054208,19.272975
min,2024-07-01 00:00:00,-399.000000
25%,2024-12-01 00:00:00,2.000000
50%,2025-05-03 00:00:00,7.000000
75%,2025-09-15 00:00:00,20.000000
max,2025-12-28 00:00:00,1153.000000
std,NaN,41.076610


New National Data Description


/Users/Ben.Pharris/Documents/Code Gallery/Media Measurement/modules/data_cleaner.py:148: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df['Date'] = pd.to_datetime(cleaned_df['Date'])


,Date,Spend
count,42876,42876.000000
mean,2025-06-24 10:51:23.123425792,15676.849461
min,2025-01-01 00:00:00,0.010000
25%,2025-04-03 00:00:00,652.287500
50%,2025-06-16 00:00:00,3240.990000
75%,2025-09-16 00:00:00,14000.820000
max,2025-12-22 00:00:00,586732.980000
std,NaN,37845.139566


All data has been cleaned and saved successfully.


# The assertation issue is due to the inclusion of sales channels --- 

In [6]:
# -- Merge datasets, build master panel

from modules.merge_data import create_master_panel
create_master_panel()

/Users/Ben.Pharris/Documents/Code Gallery/Media Measurement/modules/merge_data.py:70: RankWarning: Polyfit may be poorly conditioned
  b, a = np.polyfit(x, y, 1)  # slope b, intercept a
/Users/Ben.Pharris/Documents/Code Gallery/Media Measurement/modules/merge_data.py:70: RankWarning: Polyfit may be poorly conditioned
  b, a = np.polyfit(x, y, 1)  # slope b, intercept a
/Users/Ben.Pharris/Documents/Code Gallery/Media Measurement/modules/merge_data.py:70: RankWarning: Polyfit may be poorly conditioned
  b, a = np.polyfit(x, y, 1)  # slope b, intercept a
/Users/Ben.Pharris/Documents/Code Gallery/Media Measurement/modules/merge_data.py:70: RankWarning: Polyfit may be poorly conditioned
  b, a = np.polyfit(x, y, 1)  # slope b, intercept a
/Users/Ben.Pharris/Documents/Code Gallery/Media Measurement/modules/merge_data.py:70: RankWarning: Polyfit may be poorly conditioned
  b, a = np.polyfit(x, y, 1)  # slope b, intercept a
/Users/Ben.Pharris/Documents/Code Gallery/Media Measurement/modules/me

Using intersecting date range (excluding opensignal lag at end): 2024-07-01 to 2025-12-22
Imputing OpenSignal from 2025-08-01 → 2025-12-01


/Users/Ben.Pharris/Documents/Code Gallery/Media Measurement/modules/merge_data.py:70: RankWarning: Polyfit may be poorly conditioned
  b, a = np.polyfit(x, y, 1)  # slope b, intercept a
/Users/Ben.Pharris/Documents/Code Gallery/Media Measurement/modules/merge_data.py:70: RankWarning: Polyfit may be poorly conditioned
  b, a = np.polyfit(x, y, 1)  # slope b, intercept a
/Users/Ben.Pharris/Documents/Code Gallery/Media Measurement/modules/merge_data.py:70: RankWarning: Polyfit may be poorly conditioned
  b, a = np.polyfit(x, y, 1)  # slope b, intercept a
/Users/Ben.Pharris/Documents/Code Gallery/Media Measurement/modules/merge_data.py:70: RankWarning: Polyfit may be poorly conditioned
  b, a = np.polyfit(x, y, 1)  # slope b, intercept a
/Users/Ben.Pharris/Documents/Code Gallery/Media Measurement/modules/merge_data.py:70: RankWarning: Polyfit may be poorly conditioned
  b, a = np.polyfit(x, y, 1)  # slope b, intercept a
/Users/Ben.Pharris/Documents/Code Gallery/Media Measurement/modules/me

❌ Assertion FAILED for: Instrument_Affiliate
Series are different

Series values are different (17.03704 %)
[index]: [2024-07-01T00:00:00.000000000, 2024-07-02T00:00:00.000000000, 2024-07-03T00:00:00.000000000, 2024-07-04T00:00:00.000000000, 2024-07-05T00:00:00.000000000, 2024-07-06T00:00:00.000000000, 2024-07-07T00:00:00.000000000, 2024-07-08T00:00:00.000000000, 2024-07-09T00:00:00.000000000, 2024-07-10T00:00:00.000000000, 2024-07-11T00:00:00.000000000, 2024-07-12T00:00:00.000000000, 2024-07-13T00:00:00.000000000, 2024-07-14T00:00:00.000000000, 2024-07-15T00:00:00.000000000, 2024-07-16T00:00:00.000000000, 2024-07-17T00:00:00.000000000, 2024-07-18T00:00:00.000000000, 2024-07-19T00:00:00.000000000, 2024-07-20T00:00:00.000000000, 2024-07-21T00:00:00.000000000, 2024-07-22T00:00:00.000000000, 2024-07-23T00:00:00.000000000, 2024-07-24T00:00:00.000000000, 2024-07-25T00:00:00.000000000, 2024-07-26T00:00:00.000000000, 2024-07-27T00:00:00.000000000, 2024-07-28T00:00:00.000000000, 2024-07-29T00:

In [7]:
# --- Load in cleaned master dataset

master = pd.read_parquet(os.getenv("MASTER_DATA_PATH"))

# Attaching Other Controls

#### This breaks the beautiful modular approach, but there's a lot that needs to get fixed in here.

#### Biggest issue: _The FRED API call for the specific indices is currently being built_

- `Media Measurement/modules/macro_tools_f.py.old` is the deprecated vibe-code version
- `Media Measurement/modules/macro_tools.ipynb` is the work-in-progress new version that should eventually be modularized to a `.py` import

### The following blocks attach controls which include:

- Macroeconomic Indices:
  - **30-day lagged PCE:** PCE or personal consumption expenditure is an index of US consumer spending often used to model inflation. It is used in place of CPI to help address measurement error.
  - **7-day lagged WEI:** WEI is the weekly economic index and is a high frequency measure of economic activity, again designed to achieve the same result as CPI but more reflective of average consumer experience.
- Reverse Causality Controls:
  - **Lagged GAs:** In order to address the specific perforance issue of business performance driving investment, we use lagged outcomes to control for reverse causality. A 7 day lag is the default specification.

In [8]:
# --- Economic Controls

from modules.macro_tools import attach_macro_controls
master = attach_macro_controls(
    master,
    date_col="Date",
    lag_pce_days=30,   # conservative for PCE
    lag_wei_days=7,    # weekly cadence
    standardize=True
)

ModuleNotFoundError: No module named 'modules.macro_tools'

In [ ]:
# --- Reverse Causality Controls

from modules.reverse_causality_controls import create_lagged_features
master = create_lagged_features(
    master,
    target_col = 'Total_GAs',
    lag_days = [7],
    date_col = 'Date',
    group_col = None,
    dropna = True
)

In [ ]:
# --- Pickle the Final dataset to parquet for exporting, databricks, and analysis
master.to_parquet('./data/master.parquet')